<!-- DRAFT: placeholder prose; the author will rewrite this. -->
# 04 · Retrieval: six methods, one AQL query each
M1 BM25 · M2 vector · M3 vector + date/state filter · M4–M6 the same graph query over structural, semantic or both edge sets. Two questions from the gold set: q22, where the graph adds the second article a customer needs, and q42, where it pushes a correct one out.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, '..'); sys.path.insert(0, '../scripts')
from cookbook.db import get_db
from cookbook.load import load_gold
from cookbook.retrieve import METHODS, retrieve
db = get_db()
gold = {q['id']: q for q in load_gold()}
def show(qid, k=5):
    q = gold[qid]
    print(q['question'], '\ngold:', q['gold_keys'], '\n')
    for m, (name, _) in METHODS.items():
        rows = retrieve(db, m, q, k=k)
        print(f'{m} {name:22}', [r['key'] + (' *' if r['key'] in q['gold_keys'] else '') for r in rows])
        for r in rows:
            if r.get('via') and r['key'] in q['gold_keys']:
                print('      reached through', r['via'])

<!-- DRAFT: placeholder prose; the author will rewrite this. -->
The graph query itself (printed with its bind variables):

In [2]:
retrieve(db, 'M4', gold['q22'], k=5, show=True)[:2]

W1118 17:23:41.617000 13537 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AQL> // 1. M3's top 10 are the candidates; the first 5 are entry points
     LET cands = (
       FOR d IN articles
         FILTER d.effective_from <= @as_of AND (d.effective_to == null OR d.effective_to >= @as_of)
           AND (@state == null OR LENGTH(d.states) == 0 OR @state IN d.states)
         LET approx = APPROX_NEAR_COSINE(d.embedding, @qv)
         SORT approx DESC
         LIMIT @k
         // rank with the index, score with the exact cosine (the two differ by ~1e-8, I-019)
         RETURN { key: d._key, id: d._id, score: COSINE_SIMILARITY(d.embedding, @qv), via: null }
     )
     // 2. walk the graph from each entry over the allowed edges (@edges): to a linked article (1 hop), or
     //    through a shared form, lender, lessor or concept (2-3 hops). PRUNE stops at the first article, so no
     //    path goes through one, and at a concept so generic that it would connect everything.
     LET neighbours = (
       FOR entry IN SLICE(cands, 0, @n_entry)
         FOR v, ed

[{'key': 'duplicate_title_north_carolina__2022',
  'score': 0.6945057674636188,
  'via': None},
 {'key': 'registration_expired__2022',
  'score': 0.5767557917728692,
  'via': None}]

In [3]:
show('q22')

My offer expired while I was waiting for a duplicate title from North Carolina. Do I have to start over, and will the price be the same? 
gold: ['duplicate_title_north_carolina__2022', 'offer_validity__2026'] 

M1 BM25                   ['duplicate_title_north_carolina__2022 *', 'lost_title__2022', 'selling_without_buying__2022', 'registration_expired__2022', 'duplicate_title_georgia__2022']
M2 Vector                 ['duplicate_title_north_carolina__2022 *', 'registration_expired__2022', 'lost_title__2022', 'buy_title_northwind_finance__2022', 'duplicate_title_texas__2022']
M3 Vector + filter        ['duplicate_title_north_carolina__2022 *', 'registration_expired__2022', 'lost_title__2022', 'buy_title_northwind_finance__2022', 'selling_with_loan__2022']
M4 Graph: structural      ['duplicate_title_north_carolina__2022 *', 'registration_expired__2022', 'lost_title__2022', 'offer_validity__2026 *', 'buy_title_northwind_finance__2022']
      reached through duplicate_title_north_carolina_

M6 Graph: both layers     ['duplicate_title_north_carolina__2022 *', 'registration_expired__2022', 'lost_title__2022', 'offer_validity__2026 *', 'buy_title_northwind_finance__2022']
      reached through duplicate_title_north_carolina__2022 > offer_validity__2026


In [4]:
show('q42')

I want to sell my leased car using home pickup. I'm in Texas. What do I need to have ready? 
gold: ['home_pickup__2026', 'leased_car__2022', 'appointment_checklist__2026'] 

M1 BM25                   ['home_pickup__2026 *', 'selling_without_buying__2022', 'home_pickup__2022', 'how_selling_works__2022', 'trade_in_vs_sell__2026']


M2 Vector                 ['home_pickup__2026 *', 'home_pickup__2022', 'how_selling_works__2022', 'leased_car__2022 *', 'how_selling_works__2026']
M3 Vector + filter        ['home_pickup__2026 *', 'leased_car__2022 *', 'how_selling_works__2026', 'appointment_checklist__2026 *', 'selling_without_buying__2022']
M4 Graph: structural      ['home_pickup__2026 *', 'appointment_checklist__2026 *', 'getting_paid__2026', 'how_selling_works__2026', 'non_running_cars__2026']
      reached through home_pickup__2026 > appointment_checklist__2026
M5 Graph: semantic        ['home_pickup__2026 *', 'leased_car__2022 *', 'how_selling_works__2026', 'appointment_checklist__2026 *', 'selling_without_buying__2022']


M6 Graph: both layers     ['home_pickup__2026 *', 'appointment_checklist__2026 *', 'getting_paid__2026', 'how_selling_works__2026', 'non_running_cars__2026']
      reached through home_pickup__2026 > appointment_checklist__2026
